# API Pipeline Simulation

Simulates the full backend request pipeline using **mock API responses** — no real API keys needed.

The mock data matches the exact response shapes returned by:
- **Nominatim** (reverse geocoding: lat/lng → ZIP + state)
- **Zyla Gas Price Locator** (station prices by ZIP)
- **OSRM** (driving distances and durations)

This validates that the pipeline wiring is correct end-to-end before real API credentials are available.

In [ ]:
import sys, os

# Works whether Jupyter is launched from Gas App/ or Gas App/notebooks/
_cwd = os.getcwd()
_root = _cwd if os.path.exists(os.path.join(_cwd, 'costcalc.py')) else os.path.dirname(_cwd)
sys.path.insert(0, _root)
print(f'Project root: {_root}')

import json
import pandas as pd

from costcalc import VehicleParams, Station, recommend

print('Setup complete.')

## Step 1 — Mock API Responses

Each dict below mirrors the exact JSON shape the real API would return.

In [ ]:
# --- Nominatim reverse geocode response ---
MOCK_NOMINATIM = {
    'place_id': 298765,
    'display_name': 'San Francisco, California, United States',
    'address': {
        'city':             'San Francisco',
        'county':           'San Francisco County',
        'state':            'California',
        'ISO3166-2-lvl4':   'US-CA',
        'country':          'United States',
        'country_code':     'us',
        'postcode':         '94102',
    }
}

# --- Zyla /get+pices response (station prices for ZIP 94102) ---
MOCK_ZYLA_PRICES = {
    'stations': [
        {'id': 'zyla-001', 'name': 'Chevron',  'price': '4.29'},
        {'id': 'zyla-002', 'name': 'Shell',    'price': '4.41'},
        {'id': 'zyla-003', 'name': 'Arco',     'price': '4.15'},
        {'id': 'zyla-004', 'name': '76',        'price': '4.35'},
        {'id': 'zyla-005', 'name': 'Valero',   'price': '4.09'},
    ]
}

# --- Zyla /station+data responses (one per station ID) ---
MOCK_ZYLA_DETAILS = {
    'zyla-001': {'name': 'Chevron', 'address': {'street': '100 Market St',  'city': 'San Francisco', 'state': 'CA'}, 'coordinates': {'lat': 37.7935, 'lng': -122.3964}},
    'zyla-002': {'name': 'Shell',   'address': {'street': '200 Mission St', 'city': 'San Francisco', 'state': 'CA'}, 'coordinates': {'lat': 37.7885, 'lng': -122.3994}},
    'zyla-003': {'name': 'Arco',    'address': {'street': '800 Bryant St',  'city': 'San Francisco', 'state': 'CA'}, 'coordinates': {'lat': 37.7753, 'lng': -122.4033}},
    'zyla-004': {'name': '76',       'address': {'street': '1 Cesar Chavez', 'city': 'San Francisco', 'state': 'CA'}, 'coordinates': {'lat': 37.7488, 'lng': -122.4062}},
    'zyla-005': {'name': 'Valero',  'address': {'street': '2599 Mission St','city': 'San Francisco', 'state': 'CA'}, 'coordinates': {'lat': 37.7526, 'lng': -122.4186}},
}

# --- OSRM Table API response ---
# First coordinate is origin; indices 1-5 are the five stations.
# durations in seconds, distances in meters.
MOCK_OSRM = {
    'code': 'Ok',
    'durations': [[0.0, 240.0, 360.0, 780.0, 1020.0, 900.0]],
    'distances': [[0.0, 1450.0, 2300.0, 5800.0, 8200.0, 7100.0]],
}

print('Mock responses defined.')
print(f'  Nominatim ZIP: {MOCK_NOMINATIM["address"]["postcode"]}, state: {MOCK_NOMINATIM["address"]["ISO3166-2-lvl4"].split("-")[1]}')
print(f'  Zyla stations: {len(MOCK_ZYLA_PRICES["stations"])}')
print(f'  OSRM distances: {MOCK_OSRM["distances"][0][1:]} meters')

## Step 2 — Run the Pipeline

Replicate exactly what `backend/routers/recommend.py` does at runtime, using the mock responses above.

In [ ]:
METERS_TO_MILES = 1 / 1609.344

# --- Step A: Parse geocoding result ---
addr = MOCK_NOMINATIM['address']
zip_code = addr['postcode']
state    = addr['ISO3166-2-lvl4'].split('-')[1]
print(f'A. Location: ZIP {zip_code}, state {state}')

# --- Step B: Parse Zyla prices + station details ---
station_records = []
for entry in MOCK_ZYLA_PRICES['stations']:
    detail  = MOCK_ZYLA_DETAILS[entry['id']]
    coords  = detail['coordinates']
    addr_d  = detail['address']
    station_records.append({
        'id':    entry['id'],
        'name':  detail['name'],
        'address': f"{addr_d['street']}, {addr_d['city']}, {addr_d['state']}",
        'lat':   coords['lat'],
        'lng':   coords['lng'],
        'price_per_gallon': float(entry['price']),
    })
print(f'B. Parsed {len(station_records)} stations with prices')

# --- Step C: Attach OSRM driving distances ---
durations = MOCK_OSRM['durations'][0]
distances = MOCK_OSRM['distances'][0]

for i, rec in enumerate(station_records):
    rec['distance_miles']     = distances[i + 1] * METERS_TO_MILES
    rec['drive_time_minutes'] = durations[i + 1] / 60
print(f'C. Attached driving distances (OSRM)')

# --- Step D: Build Station objects ---
station_objs = [
    Station(
        name=r['name'],
        price_per_gallon=r['price_per_gallon'],
        distance_miles=r['distance_miles'],
        drive_time_minutes=r['drive_time_minutes'],
    )
    for r in station_records
]
meta_map = {id(s): r for s, r in zip(station_objs, station_records)}
print(f'D. Built {len(station_objs)} Station objects')

# --- Step E: Run cost engine ---
vehicle = VehicleParams(mpg=28, tank_current=4.5, tank_capacity=13.2)
result  = recommend(station_objs, vehicle)
print(f'E. Cost engine complete — winner: {result.best_station.station.name}')

## Step 3 — Recommendation Output

In [ ]:
print('=' * 55)
print('  RECOMMENDATION')
print('=' * 55)

best = result.best_station
meta = meta_map[id(best.station)]
print(f'  Station:          {best.station.name}')
print(f'  Address:          {meta["address"]}')
print(f'  Price per gallon: ${best.station.price_per_gallon:.3f}')
print(f'  Distance:         {best.station.distance_miles:.2f} miles')
print(f'  Drive time:       {best.station.drive_time_minutes:.1f} min')
print(f'  Gallons bought:   {best.gallons_purchased:.3f} gal')
print(f'  Effective cost:   ${best.effective_cost:.2f}')
print(f'  Note:             {result.note}')

print()
print('  FULL RANKING')
print('-' * 55)

rows = []
for r in result.all_results:
    rows.append({
        'Rank':        r.rank,
        'Station':     r.station.name,
        'Price/gal':   f'${r.station.price_per_gallon:.3f}',
        'Distance':    f'{r.station.distance_miles:.2f} mi',
        'Drive time':  f'{r.station.drive_time_minutes:.1f} min' if r.station.drive_time_minutes else 'N/A',
        'Eff. Cost':   f'${r.effective_cost:.2f}' if r.reachable else 'unreachable',
        'vs Best':     f'${r.savings_vs_best:.2f}' if r.reachable else '—',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Step 4 — Verify Response Shape Matches API Schema

The response the router sends back should match `RecommendResponse` in `backend/schemas.py`. Verify the field names here match what the schema expects.

In [ ]:
def build_api_response(result, meta_map, state, zip_code):
    """Build the dict that the router returns — matches RecommendResponse schema."""
    def station_out(r):
        m = meta_map[id(r.station)]
        return {
            'name':                  r.station.name,
            'address':               m.get('address', ''),
            'latitude':              m['lat'],
            'longitude':             m['lng'],
            'price_per_gallon':      r.station.price_per_gallon,
            'distance_miles':        round(r.station.distance_miles, 3),
            'drive_time_minutes':    r.station.drive_time_minutes,
            'effective_cost':        round(r.effective_cost, 2),
            'gallons_purchased':     round(r.gallons_purchased, 3),
            'fuel_burned_in_transit':round(r.fuel_burned_in_transit, 3),
            'rank':                  r.rank,
            'savings_vs_best':       round(r.savings_vs_best, 2),
            'reachable':             r.reachable,
        }

    prices = [r.station.price_per_gallon for r in result.all_results if r.reachable]
    return {
        'best_station':    station_out(result.best_station),
        'all_stations':    [station_out(r) for r in result.all_results],
        'note':            result.note,
        'region_avg_price':round(sum(prices) / len(prices), 3),
        'state':           state,
        'tank_is_full':    result.tank_is_full,
    }

response = build_api_response(result, meta_map, state, zip_code)
print(json.dumps(response, indent=2))